# Extended RFM Customer Segmentation
**Single Canonical Dataset** - Online Retail (UCI/Kaggle)
Thesis: Batch 62, Dept of CSE, City University, Bangladesh

In [1]:
import os, warnings, numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score
warnings.filterwarnings('ignore')
print('Libraries loaded.')

Libraries loaded.


In [2]:
df_raw = pd.read_csv('data/online_retail.csv', encoding='ISO-8859-1')
df = df_raw.dropna(subset=['CustomerID']).copy()
df['InvoiceNo'] = df['InvoiceNo'].astype(str)
df = df[~df['InvoiceNo'].str.startswith('C')]
df = df[(df['Quantity']>0)&(df['UnitPrice']>0)].drop_duplicates()
df['TotalPrice'] = df['Quantity']*df['UnitPrice']
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(f'Cleaned rows: {len(df):,}  |  Unique customers: {df.CustomerID.nunique():,}')

Cleaned rows: 392,692  |  Unique customers: 4,338


In [3]:
snap = df['InvoiceDate'].max() + pd.Timedelta(days=1)
def ipt(dates):
    u = np.sort(dates.dt.floor('D').unique())
    return 0.0 if len(u)<=1 else float(np.mean(np.diff(u)/np.timedelta64(1,'D')))
rfm = df.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x:(snap-x.max()).days),
    Frequency=('InvoiceNo','nunique'),
    Monetary=('TotalPrice','sum'),
    InterPurchase_Time=('InvoiceDate', ipt),
    Basket_Depth=('StockCode','nunique'))
fcols=['Recency','Frequency','Monetary','InterPurchase_Time','Basket_Depth']
rfm_sc = pd.DataFrame(StandardScaler().fit_transform(np.log1p(rfm[fcols])),
                      index=rfm.index, columns=fcols)
print(rfm.describe().round(2))

       Recency  Frequency   Monetary  InterPurchase_Time  Basket_Depth
count  4338.00    4338.00    4338.00             4338.00       4338.00
mean     92.54       4.27    2048.69               50.44         61.50
std     100.01       7.70    8985.23               65.24         85.37
min       1.00       1.00       3.75                0.00          1.00
25%      18.00       1.00     306.48                0.00         16.00
50%      51.00       2.00     668.57               31.00         35.00
75%     142.00       5.00    1660.60               73.00         77.00
max     374.00     209.00  280206.02              366.00       1787.00


In [4]:
evals=[]
for k in range(2,11):
    km=KMeans(n_clusters=k,init='k-means++',random_state=42,n_init=10)
    l=km.fit_predict(rfm_sc)
    evals.append({'k':k,'WCSS':km.inertia_,
                  'Silhouette':silhouette_score(rfm_sc,l),
                  'DBI':davies_bouldin_score(rfm_sc,l)})
ev=pd.DataFrame(evals); print(ev.to_string(index=False))

 k         WCSS  Silhouette      DBI
 2 11535.093858    0.415671 0.944034
 3  8308.116920    0.364606 1.061266
 4  7264.356599    0.311300 1.244167
 5  6408.695833    0.253913 1.333817
 6  5762.248439    0.253118 1.282132
 7  5288.942970    0.253854 1.210391
 8  4876.799401    0.239062 1.208296
 9  4562.988200    0.237953 1.252093
10  4334.620600    0.235406 1.237836


In [5]:
km4=KMeans(n_clusters=4,init='k-means++',random_state=42,n_init=10)
rfm['Cluster']=km4.fit_predict(rfm_sc)
order=rfm.groupby('Cluster')['Monetary'].mean().sort_values(ascending=False).index.tolist()
lmap={order[0]:'Champions (VIP Loyalists)',order[1]:'Loyal & Promising Regulars',
      order[2]:'High-Potential / At-Risk Guests',order[3]:'Hibernating / Lost Guests'}
rfm['Segment']=rfm['Cluster'].map(lmap)
print(rfm.groupby('Segment').size())

Segment
Champions (VIP Loyalists)           515
Hibernating / Lost Guests          1572
High-Potential / At-Risk Guests    1071
Loyal & Promising Regulars         1180
dtype: int64


All agents complete. Outputs are in the working directory.